In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append("../..")

from src.classifical_features import build_monthly_features
from src.configs.settings import Settings
from src.visualization.visualization import (
    plot_best_predictions,
    plot_overall_metrics_comparison,
    plot_panel_metrics_distributions,
    plot_worst_predictions,
)

settings = Settings()

In [ ]:
data_dir = Path("../../data")

In [ ]:
features_df = pd.read_csv(data_dir / "processed" / "mirrors_clusters.csv")
mirrors_ts_df = pd.read_csv(data_dir / "processed" / "filtered_mirrors_ts.csv")

In [ ]:
features = build_monthly_features(mirrors_ts_df, settings)
features["panel_id"] = features[settings.columns.id].astype(str)

In [ ]:
from src.catboost_utilities.evaluate import evaluate_catboost
from src.catboost_utilities.train import train_catboost
from src.custom_types import CatBoostParameters
from src.model_selection import temporal_panel_split_by_size

regression_splits = temporal_panel_split_by_size(
    features,
    panel_column='article',
    time_column='date',
    test_size=3,
    val_size=None
)

In [ ]:
from src.data_processing import scale_panel_splits

scale = not settings.downstream.round_predictions
if scale:
    splits = scale_panel_splits(
        (data for _, data in regression_splits.splits),
        panel_column=settings.columns.id,
        target_columns=settings.columns.main_target,
        apply_log=settings.preprocessing.apply_log,
    )
else:
    splits = regression_splits

In [ ]:
catboost_params = CatBoostParameters(
    iterations=500,
    learning_rate=0.05,
    depth=3,
    l2_leaf_reg=10,
    subsample=0.8,
    rsm=0.8,
    random_seed=settings.random_state,
    verbose=50,
    loss_function="RMSE",
)

catboost_model = train_catboost(
    train_df=splits.train,
    val_df=splits.val,
    params=catboost_params,
    settings=settings,
)

results = evaluate_catboost(
    model=catboost_model,
    splits=splits,
    settings=settings,
    scalers=None,
)

In [ ]:
overall_df = results.get_overall_metrics_df()
panel_df = results.get_panel_metrics_df()


plot_overall_metrics_comparison(results)

plot_panel_metrics_distributions(
    results=results,
    metrics_to_plot=["mape", "rmse", "mae", "r2"],
)

In [ ]:
plot_best_predictions(
    results=results,
    n_best=5,
    metric="rmse",
    split_name="test",
)

In [ ]:
plot_worst_predictions(
    results=results,
    n_worst=5,
    metric="rmse",
    split_name="test",
)

In [ ]:
feature_importance = catboost_model.get_feature_importance()
feature_names = regression_splits.train.drop(
    columns=[settings.columns.main_target, settings.columns.id, settings.columns.date]
).columns

importance_df = pd.DataFrame(
    {
        "feature": feature_names,
        "importance": feature_importance,
    }
).sort_values("importance", ascending=False)

importance_df